# HomeWork exercise assignment

In [ ]:
import ollama
import requests
from bs4 import BeautifulSoup
from IPython.display import Markdown, display
import logging

# Setup logging
logging.basicConfig(level=logging.INFO)

# -------------------------------
# CONFIG
# -------------------------------
MODEL_NAME = "gemma4:31b-cloud"
MAX_CHUNK_SIZE = 3000  # tokens safety approx


# -------------------------------
# FETCH WEBSITE CONTENT
# -------------------------------
def fetch_website_contents(url: str) -> str:
    try:
        headers = {
            "User-Agent": "Mozilla/5.0 (compatible; AI-Summarizer/1.0)"
        }

        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")

        # Remove unwanted tags
        for tag in soup(["script", "style", "nav", "footer", "header", "aside"]):
            tag.extract()

        text = soup.get_text(separator=" ", strip=True)

        logging.info("Website content fetched successfully.")
        return text

    except requests.exceptions.RequestException as e:
        logging.error(f"Request failed: {e}")
        return ""


# -------------------------------
# SPLIT TEXT INTO CHUNKS
# -------------------------------
def chunk_text(text: str, chunk_size: int = MAX_CHUNK_SIZE):
    return [text[i:i + chunk_size] for i in range(0, len(text), chunk_size)]


# -------------------------------
# CREATE MESSAGES
# -------------------------------
def messages_for(text_chunk: str):
    return [
        {
            "role": "system",
            "content": "You are a helpful AI tutor. I want you to give everything every topics which are given in the website with the bullet points as well. Generate a roadmap and study plan for all those topics. And practice questions for every topics in the sheet."
        },
        {
            "role": "user",
            "content": text_chunk
        }
    ]


# -------------------------------
# SUMMARIZE SINGLE CHUNK
# -------------------------------
def summarize_chunk(text_chunk: str) -> str:
    try:
        response = ollama.chat(
            model=MODEL_NAME,
            messages=messages_for(text_chunk)
        )
        return response["message"]["content"]

    except Exception as e:
        logging.error(f"Summarization error: {e}")
        return ""


# -------------------------------
# MAIN SUMMARIZE FUNCTION
# -------------------------------
def summarize(url: str) -> str:
    text = fetch_website_contents(url)

    if not text:
        return "❌ Failed to fetch website content."

    chunks = chunk_text(text)

    logging.info(f"Total chunks: {len(chunks)}")

    summaries = []
    for i, chunk in enumerate(chunks):
        logging.info(f"Processing chunk {i + 1}/{len(chunks)}...")
        summary = summarize_chunk(chunk)
        summaries.append(summary)

    # Final summary refinement
    final_summary = summarize_chunk(" ".join(summaries))

    return final_summary


# -------------------------------
# DISPLAY
# -------------------------------
def display_summary(url: str):
    summary = summarize(url)
    display(Markdown(summary))


# -------------------------------
# RUN
# -------------------------------
if __name__ == "__main__":
    display_summary("https://neetcode.io/practice/practice/neetcode150")